# Basilisk quickstart — a real spacecraft simulation in your browser tab

This notebook runs [Basilisk](https://avslab.github.io/basilisk/), the open-source astrodynamics
simulation framework from the AVS Lab (University of Colorado Boulder), at **full native fidelity** —
the same C++ flight-dynamics engine used in research and mission work. No local install needed:
the next cell pulls the official prebuilt package (~85 MB) in a couple of minutes.

Part of the [BSDS project](https://github.com/JPWPWhitney/BSDS) — Basilisk missions in the browser.


In [ ]:
%pip install bsk==2.11.1 matplotlib --quiet

## Simulate one orbit

We build the classic minimal mission: a spacecraft in low Earth orbit under point-mass Earth
gravity, propagated with Basilisk's RK4 integrator, with a message recorder capturing the
trajectory. This mirrors Basilisk's official `scenarioBasicOrbit` tutorial.


In [ ]:
import numpy as np
from Basilisk.utilities import SimulationBaseClass, macros, orbitalMotion, simIncludeGravBody
from Basilisk.simulation import spacecraft

# --- Build the simulation ---------------------------------------------------
scSim = SimulationBaseClass.SimBaseClass()
dynProcess = scSim.CreateNewProcess("dynamicsProcess")
timestep = macros.sec2nano(10.0)
dynProcess.addTask(scSim.CreateNewTask("dynamicsTask", timestep))

scObject = spacecraft.Spacecraft()
scObject.ModelTag = "bsds-sat"
scSim.AddModelToTask("dynamicsTask", scObject)

gravFactory = simIncludeGravBody.gravBodyFactory()
earth = gravFactory.createEarth()
earth.isCentralBody = True
gravFactory.addBodiesTo(scObject)

# --- Initial orbit: 500 km circular-ish LEO ---------------------------------
mu = earth.mu
oe = orbitalMotion.ClassicElements()
oe.a = 6878.137e3      # semi-major axis [m]
oe.e = 0.01
oe.i = 51.6 * macros.D2R
oe.Omega = 48.2 * macros.D2R
oe.omega = 347.8 * macros.D2R
oe.f = 0.0
rN, vN = orbitalMotion.elem2rv(mu, oe)
scObject.hub.r_CN_NInit = rN
scObject.hub.v_CN_NInit = vN

# --- Record and run one orbit ------------------------------------------------
period = 2 * np.pi * np.sqrt(oe.a**3 / mu)
rec = scObject.scStateOutMsg.recorder(macros.sec2nano(10.0))
scSim.AddModelToTask("dynamicsTask", rec)

scSim.InitializeSimulation()
scSim.ConfigureStopTime(macros.sec2nano(1.05 * period))
scSim.ExecuteSimulation()

t = rec.times() * macros.NANO2SEC
r = np.array(rec.r_BN_N)
alt_km = (np.linalg.norm(r, axis=1) - 6378.137e3) / 1000.0
print(f"samples={len(t)}  period={period/60:.1f} min  alt range: {alt_km.min():.1f}-{alt_km.max():.1f} km")
assert len(t) > 100 and np.all(np.isfinite(r))
print("SIM OK")


## Plot the mission

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
r_km = r / 1000.0
theta = np.linspace(0, 2 * np.pi, 200)
Re = 6378.137
ax1.fill(Re * np.cos(theta), Re * np.sin(theta), color="#3b6aa0", alpha=0.6, label="Earth")
ax1.plot(r_km[:, 0], r_km[:, 1], color="#d1495b", lw=1.5, label="orbit")
ax1.set_aspect("equal"); ax1.set_xlabel("x [km]"); ax1.set_ylabel("y [km]")
ax1.set_title("Orbit (inertial x-y projection)"); ax1.legend()
ax2.plot(t / 60.0, alt_km, color="#d1495b")
ax2.set_xlabel("time [min]"); ax2.set_ylabel("altitude [km]")
ax2.set_title("Altitude vs time"); ax2.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Where to go from here

- Change `oe.a`, `oe.e`, `oe.i` above and re-run — you're flying the mission.
- Basilisk's full example catalog: [Example Scripts](https://avslab.github.io/basilisk/examples/index.html)
  (`pip install "bsk[all,examples]"` then run `bskExamples` to download them).
- The BSDS repo ([JPWPWhitney/BSDS](https://github.com/JPWPWhitney/BSDS)) has a one-click
  **GitHub Codespaces** workbench — a persistent browser VS Code with Basilisk preinstalled.

*Powered by Basilisk — © Autonomous Vehicle Systems Lab, University of Colorado Boulder, ISC License.*
